In [ ]:
# ============================================================
# RSNA Knee MRI - LLM Pseudo Labels V2
# Step 1: Setup
# ============================================================

import os
import json
import time
import pandas as pd
import numpy as np

from pathlib import Path
from kaggle_secrets import UserSecretsClient

# -------------------------
# Paths
# -------------------------

DATA_DIR = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

OUTPUT_DIR = Path(
    "/kaggle/working/llm_pseudo_labels_v2"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# -------------------------
# Competition targets
# -------------------------

TARGET_COLS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

print("Setup complete ✓")
print("Output directory:", OUTPUT_DIR)

In [2]:
# ============================================================
# Step 2: Load competition reports
# ============================================================

train = pd.read_csv(
    DATA_DIR / "train.csv"
)

print("train shape:", train.shape)
print("Unique studies:", train["StudyInstanceUID"].nunique())
print("Missing reports:", train["Report"].isna().sum())

# Identify the 58 gold and 4,349 unlabelled studies

gold_mask = train[TARGET_COLS].notna().all(axis=1)

gold_df = train.loc[gold_mask].copy().reset_index(drop=True)
unlabelled_df = train.loc[~gold_mask].copy().reset_index(drop=True)

print("\nGold studies:", len(gold_df))
print("Unlabelled studies:", len(unlabelled_df))

assert len(gold_df) == 58
assert len(unlabelled_df) == 4349
assert train["Report"].isna().sum() == 0

print("\nDataset verification passed ✓")

train shape: (4407, 14)
Unique studies: 4407
Missing reports: 0

Gold studies: 58
Unlabelled studies: 4349

Dataset verification passed ✓


In [3]:
# ============================================================
# Step 3: Connect OpenAI API
# ============================================================

!pip install -q openai

from openai import OpenAI

# Read API key securely from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("OPENAI_API_KEY")

# Create client
client = OpenAI(api_key=api_key)

print("OpenAI API connected ✓")

OpenAI API connected ✓


In [4]:
# ============================================================
# Step 4: Define Soft Label V1 output specification
# ============================================================

SOFT_LABEL_SYSTEM_PROMPT = """
You are analysing knee MRI radiology reports for a multilabel
medical-imaging dataset.

For each of the 12 targets, estimate the probability that the
dataset's official binary label is 1.

Important:
- Return a continuous score from 0.0 to 1.0.
- Do not convert predictions into hard 0/1 labels.
- Distinguish confirmed abnormalities from explicitly normal findings.
- Account for negation, uncertainty, chronic/postoperative findings,
  synonyms and multilingual terminology.
- Do not assume that an unmentioned finding is definitely negative.
- Use the entire report.
- Confidence describes confidence in extracting the label from the
  report, not the severity of the condition.

Score guidance:
- 0.00–0.10: clearly negative
- 0.10–0.35: probably negative or not clearly supported
- 0.35–0.65: uncertain, ambiguous or insufficiently described
- 0.65–0.90: probably positive
- 0.90–1.00: clearly positive

Confidence guidance:
- 0.90–1.00: explicit and unambiguous evidence
- 0.60–0.89: reasonably supported but indirect or mildly uncertain
- 0.30–0.59: ambiguous, incomplete or difficult to interpret
- 0.00–0.29: insufficient evidence

Return valid JSON only, using exactly this structure:

{
  "ACL": {
    "score": 0.0,
    "confidence": 0.0,
    "evidence": ""
  },
  "MCL": {
    "score": 0.0,
    "confidence": 0.0,
    "evidence": ""
  },
  "Medial Meniscus": {
    "score": 0.0,
    "confidence": 0.0,
    "evidence": ""
  },
  "Lateral Meniscus": {
    "score": 0.0,
    "confidence": 0.0,
    "evidence": ""
  },
  "Medial OA": {
    "score": 0.0,
    "confidence": 0.0,
    "evidence": ""
  },
  "Lateral OA": {
    "score": 0.0,
    "confidence": 0.0,
    "evidence": ""
  },
  "PF OA": {
    "score": 0.0,
    "confidence": 0.0,
    "evidence": ""
  },
  "Effusion": {
    "score": 0.0,
    "confidence": 0.0,
    "evidence": ""
  },
  "Synovitis": {
    "score": 0.0,
    "confidence": 0.0,
    "evidence": ""
  },
  "Baker's": {
    "score": 0.0,
    "confidence": 0.0,
    "evidence": ""
  },
  "Contusion": {
    "score": 0.0,
    "confidence": 0.0,
    "evidence": ""
  },
  "Fracture": {
    "score": 0.0,
    "confidence": 0.0,
    "evidence": ""
  }
}
"""

SOFT_LABEL_FIELDS = ["score", "confidence", "evidence"]

print("Soft Label V1 specification created ✓")
print("Targets:", len(TARGET_COLS))
print("Required fields:", SOFT_LABEL_FIELDS)

assert len(TARGET_COLS) == 12
assert set(TARGET_COLS) == {
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
}

print("Soft Label V1 specification verified ✓")

Soft Label V1 specification created ✓
Targets: 12
Required fields: ['score', 'confidence', 'evidence']
Soft Label V1 specification verified ✓


In [5]:
# ============================================================
# Step 5: Create multilabel-stratified OOF folds
# ============================================================

!pip install -q iterative-stratification

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

N_FOLDS = 5
RANDOM_STATE = 42

X_gold = np.zeros((len(gold_df), 1))
y_gold = gold_df[TARGET_COLS].astype(int).values

fold_splitter = MultilabelStratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

gold_df["fold"] = -1

for fold, (_, val_idx) in enumerate(
    fold_splitter.split(X_gold, y_gold)
):
    gold_df.loc[val_idx, "fold"] = fold

gold_df["fold"] = gold_df["fold"].astype(int)

assert (gold_df["fold"] >= 0).all()
assert gold_df["StudyInstanceUID"].is_unique
assert len(gold_df) == 58

# Save the fixed fold assignment
FOLD_FILE = OUTPUT_DIR / "soft_label_v1_gold_5fold.csv"

gold_df[
    ["StudyInstanceUID", "fold"] + TARGET_COLS
].to_csv(FOLD_FILE, index=False)

print("Gold reports by fold:")
print(gold_df["fold"].value_counts().sort_index())

print("\nOverall positive counts:")
print(gold_df[TARGET_COLS].sum().astype(int))

print("\nPositive counts by fold:")
print(
    gold_df.groupby("fold")[TARGET_COLS]
    .sum()
    .astype(int)
    .T
)

print("\nSaved:", FOLD_FILE)
print("Leakage-free five-fold assignment created ✓")

Gold reports by fold:
fold
0    12
1    10
2    12
3    11
4    13
Name: count, dtype: int64

Overall positive counts:
ACL                 24
MCL                  9
Medial Meniscus     26
Lateral Meniscus    23
Medial OA           15
Lateral OA          11
PF OA               21
Effusion            35
Synovitis           27
Baker's             12
Contusion           19
Fracture            18
dtype: int64

Positive counts by fold:
fold              0  1   2  3  4
ACL               4  6   4  4  6
MCL               2  1   2  2  2
Medial Meniscus   5  6   5  4  6
Lateral Meniscus  4  4   5  5  5
Medial OA         3  3   3  3  3
Lateral OA        2  3   2  2  2
PF OA             4  4   4  5  4
Effusion          5  6  10  5  9
Synovitis         6  6   5  5  5
Baker's           3  3   2  2  2
Contusion         3  4   5  3  4
Fracture          3  3   6  3  3

Saved: /kaggle/working/llm_pseudo_labels_v2/soft_label_v1_gold_5fold.csv
Leakage-free five-fold assignment created ✓


In [6]:
# ============================================================
# Step 6: Select leakage-free demonstrations for each OOF fold
# ============================================================

N_DEMONSTRATIONS = 10

def select_fold_demonstrations(train_part, n_examples=10):
    """
    Greedily select gold reports covering as many target-label
    combinations as possible. Shorter reports break ties to
    reduce prompt size and API cost.
    """
    remaining = train_part.copy()
    selected_indices = []

    all_states = {
        (target, state)
        for target in TARGET_COLS
        for state in [0, 1]
    }
    covered_states = set()

    while (
        len(selected_indices) < n_examples
        and len(remaining) > 0
    ):
        best_idx = None
        best_gain = -1
        best_length = float("inf")

        for idx, row in remaining.iterrows():
            row_states = {
                (target, int(row[target]))
                for target in TARGET_COLS
            }

            gain = len(row_states - covered_states)
            report_length = len(str(row["Report"]))

            if (
                gain > best_gain
                or (
                    gain == best_gain
                    and report_length < best_length
                )
            ):
                best_idx = idx
                best_gain = gain
                best_length = report_length

        selected_indices.append(best_idx)

        selected_row = remaining.loc[best_idx]
        covered_states.update(
            (target, int(selected_row[target]))
            for target in TARGET_COLS
        )

        remaining = remaining.drop(index=best_idx)

    selected_df = train_part.loc[selected_indices].copy()

    return selected_df, covered_states


fold_demonstrations = {}

for fold in range(N_FOLDS):
    train_part = gold_df[gold_df["fold"] != fold].copy()
    val_part = gold_df[gold_df["fold"] == fold].copy()

    demonstrations, covered = select_fold_demonstrations(
        train_part,
        n_examples=N_DEMONSTRATIONS
    )

    # Leakage checks
    demo_uids = set(
        demonstrations["StudyInstanceUID"].astype(str)
    )
    val_uids = set(
        val_part["StudyInstanceUID"].astype(str)
    )

    assert demo_uids.isdisjoint(val_uids)
    assert len(demonstrations) == N_DEMONSTRATIONS

    fold_demonstrations[fold] = demonstrations

    print(
        f"Fold {fold}: "
        f"{len(val_part)} validation reports | "
        f"{len(demonstrations)} demonstrations | "
        f"{len(covered)}/24 target-label states covered | "
        f"overlap = {len(demo_uids & val_uids)}"
    )

print("\nLeakage-free OOF demonstrations created ✓")

Fold 0: 12 validation reports | 10 demonstrations | 24/24 target-label states covered | overlap = 0
Fold 1: 10 validation reports | 10 demonstrations | 24/24 target-label states covered | overlap = 0
Fold 2: 12 validation reports | 10 demonstrations | 24/24 target-label states covered | overlap = 0
Fold 3: 11 validation reports | 10 demonstrations | 24/24 target-label states covered | overlap = 0
Fold 4: 13 validation reports | 10 demonstrations | 24/24 target-label states covered | overlap = 0

Leakage-free OOF demonstrations created ✓


In [7]:
# ============================================================
# Step 7: Build structured Soft Label V1 OOF prompts
# ============================================================

from typing import Literal
from pydantic import BaseModel, Field

TargetName = Literal[
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]


class TargetSoftPrediction(BaseModel):
    target: TargetName
    score: float = Field(ge=0.0, le=1.0)
    confidence: float = Field(ge=0.0, le=1.0)
    evidence: str


class SoftLabelPrediction(BaseModel):
    predictions: list[TargetSoftPrediction]


def format_gold_demonstration(row):
    labels = {
        target: int(row[target])
        for target in TARGET_COLS
    }

    return (
        "REFERENCE REPORT:\n"
        f"{str(row['Report']).strip()}\n\n"
        "OFFICIAL DATASET LABELS:\n"
        f"{json.dumps(labels, ensure_ascii=False)}"
    )


def build_oof_prompt(report, fold):
    demonstrations = fold_demonstrations[fold]

    demonstration_text = "\n\n".join(
        format_gold_demonstration(row)
        for _, row in demonstrations.iterrows()
    )

    return f"""
The following are gold-labelled reference examples from the same
dataset. They are provided only to demonstrate the dataset-specific
annotation convention.

None of these references is the report being evaluated.

{demonstration_text}

--------------------------------------
----------------------

REPORT TO ANALYSE:

{str(report).strip()}

Analyse only the REPORT TO ANALYSE.

Return exactly one prediction for every target in TARGET_COLS.
Scores must remain continuous probabilities rather than hard labels.
Do not copy the overall label pattern from any reference example.
"""


# Build one prompt without calling the API
test_row = gold_df.iloc[0]
test_fold = int(test_row["fold"])

test_prompt = build_oof_prompt(
    report=test_row["Report"],
    fold=test_fold
)

test_demo_uids = set(
    fold_demonstrations[test_fold]["StudyInstanceUID"].astype(str)
)

assert str(test_row["StudyInstanceUID"]) not in test_demo_uids
assert "REPORT TO ANALYSE:" in test_prompt
assert len(fold_demonstrations[test_fold]) == N_DEMONSTRATIONS

print("Test validation fold:", test_fold)
print("Demonstrations:", len(fold_demonstrations[test_fold]))
print("Prompt characters:", len(test_prompt))
print("Evaluated report excluded from demonstrations ✓")
print("Structured Soft Label V1 prompt builder ready ✓")

Test validation fold: 2
Demonstrations: 10
Prompt characters: 9237
Evaluated report excluded from demonstrations ✓
Structured Soft Label V1 prompt builder ready ✓


In [8]:
# ============================================================
# Step 8: Test one Soft Label V1 API prediction
# ============================================================

MODEL_NAME = "gpt-5.6"

test_row = gold_df.iloc[0]
test_fold = int(test_row["fold"])

test_response = client.responses.parse(
    model=MODEL_NAME,
    reasoning={"effort": "low"},
    input=[
        {
            "role": "system",
            "content": SOFT_LABEL_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": build_oof_prompt(
                report=test_row["Report"],
                fold=test_fold
            )
        }
    ],
    text_format=SoftLabelPrediction
)

test_prediction = test_response.output_parsed

assert test_prediction is not None
assert len(test_prediction.predictions) == 12

returned_targets = [
    item.target
    for item in test_prediction.predictions
]

assert len(set(returned_targets)) == 12
assert set(returned_targets) == set(TARGET_COLS)

print("Model:", MODEL_NAME)
print("Study:", test_row["StudyInstanceUID"])
print("OOF fold:", test_fold)
print("Predictions returned:", len(test_prediction.predictions))
print()

for item in test_prediction.predictions:
    actual = int(test_row[item.target])

    print(
        f"{item.target:18s} "
        f"score={item.score:.3f} | "
        f"confidence={item.confidence:.3f} | "
        f"gold={actual} | "
        f"evidence={item.evidence[:100]}"
    )

print("\nSingle Soft Label V1 API test passed ✓")

Model: gpt-5.6
Study: 1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149
OOF fold: 2
Predictions returned: 12

ACL                score=0.020 | confidence=0.980 | gold=0 | evidence=Ligamentos cruzados dentro de límites normales.
MCL                score=0.020 | confidence=0.980 | gold=0 | evidence=Ligamentos colaterales dentro de límites normales.
Medial Meniscus    score=0.020 | confidence=0.990 | gold=0 | evidence=Menisco medial con morfología y señal conservadas, sin signos de rotura.
Lateral Meniscus   score=0.940 | confidence=0.960 | gold=0 | evidence=Amputación marginal del cuerpo del menisco lateral, repetida en la impresión.
Medial OA          score=0.030 | confidence=0.940 | gold=0 | evidence=Cartílagos femorotibiales sin alteraciones; la lesión condral descrita es troclear/patelofemoral.
Lateral OA         score=0.030 | confidence=0.940 | gold=0 | evidence=Cartílagos femorotibiales sin alteraciones; no se describe artrosis lateral.
PF OA              score=0.980

In [9]:
# ============================================================
# Step 9: Generate all 58 OOF Soft Label V1 predictions
# ============================================================

from tqdm.auto import tqdm

OOF_FILE = OUTPUT_DIR / "soft_label_v1_oof_predictions.csv"


def prediction_to_row(study_row, prediction):
    output_row = {
        "StudyInstanceUID": str(study_row["StudyInstanceUID"]),
        "fold": int(study_row["fold"]),
        "model": MODEL_NAME
    }

    prediction_map = {
        item.target: item
        for item in prediction.predictions
    }

    assert set(prediction_map) == set(TARGET_COLS)

    for target in TARGET_COLS:
        item = prediction_map[target]

        output_row[f"{target}_score"] = float(item.score)
        output_row[f"{target}_confidence"] = float(item.confidence)
        output_row[f"{target}_evidence"] = item.evidence
        output_row[f"{target}_gold"] = int(study_row[target])

    return output_row


# Resume from an existing checkpoint if present
if OOF_FILE.exists():
    oof_results = pd.read_csv(OOF_FILE).to_dict("records")

    completed_uids = {
        str(row["StudyInstanceUID"])
        for row in oof_results
    }

    print("Resuming completed predictions:", len(completed_uids))

else:
    oof_results = []
    completed_uids = set()

    # Reuse the paid prediction from Step 8
    if "test_prediction" in globals():
        first_result = prediction_to_row(
            test_row,
            test_prediction
        )

        oof_results.append(first_result)
        completed_uids.add(
            str(test_row["StudyInstanceUID"])
        )

        pd.DataFrame(oof_results).to_csv(
            OOF_FILE,
            index=False
        )

        print("Saved Step 8 test prediction as OOF result 1/58")


for _, row in tqdm(
    gold_df.iterrows(),
    total=len(gold_df),
    desc="Generating OOF soft labels"
):
    uid = str(row["StudyInstanceUID"])

    if uid in completed_uids:
        continue

    fold = int(row["fold"])
    last_error = None

    for attempt in range(3):
        try:
            response = client.responses.parse(
                model=MODEL_NAME,
                reasoning={"effort": "low"},
                input=[
                    {
                        "role": "system",
                        "content": SOFT_LABEL_SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": build_oof_prompt(
                            report=row["Report"],
                            fold=fold
                        )
                    }
                ],
                text_format=SoftLabelPrediction
            )

            prediction = response.output_parsed

            assert prediction is not None
            assert len(prediction.predictions) == 12

            result_row = prediction_to_row(
                row,
                prediction
            )

            oof_results.append(result_row)
            completed_uids.add(uid)

            # Save after every successful paid request
            pd.DataFrame(oof_results).to_csv(
                OOF_FILE,
                index=False
            )

            break

        except Exception as error:
            last_error = error
            wait_seconds = 2 ** attempt

            print(
                f"\nAttempt {attempt + 1}/3 failed "
                f"for {uid}: {error}"
            )

            time.sleep(wait_seconds)

    else:
        raise RuntimeError(
            f"Prediction failed after 3 attempts for {uid}: "
            f"{last_error}\n"
            f"Completed work remains saved in {OOF_FILE}"
        )


oof_df = pd.read_csv(OOF_FILE)

assert len(oof_df) == 58
assert oof_df["StudyInstanceUID"].astype(str).nunique() == 58
assert set(oof_df["fold"].astype(int)) == set(range(N_FOLDS))

print("\nOOF predictions:", len(oof_df))
print("Unique studies:", oof_df["StudyInstanceUID"].nunique())
print("Saved:", OOF_FILE)
print("All 58 OOF Soft Label V1 predictions complete ✓")

Saved Step 8 test prediction as OOF result 1/58


Generating OOF soft labels:   0%|          | 0/58 [00:00<?, ?it/s]


OOF predictions: 58
Unique studies: 58
Saved: /kaggle/working/llm_pseudo_labels_v2/soft_label_v1_oof_predictions.csv
All 58 OOF Soft Label V1 predictions complete ✓


In [10]:
# ============================================================
# Step 10: Evaluate OOF Soft Label V1 ROC-AUC
# ============================================================

from sklearn.metrics import roc_auc_score

metric_rows = []

for target in TARGET_COLS:
    gold_col = f"{target}_gold"
    score_col = f"{target}_score"
    confidence_col = f"{target}_confidence"

    y_true = oof_df[gold_col].astype(int)
    y_score = oof_df[score_col].astype(float)

    auc = roc_auc_score(y_true, y_score)

    metric_rows.append({
        "Target": target,
        "Positive": int(y_true.sum()),
        "Negative": int((1 - y_true).sum()),
        "ROC_AUC": auc,
        "MeanScore_Gold0": y_score[y_true == 0].mean(),
        "MeanScore_Gold1": y_score[y_true == 1].mean(),
        "MeanConfidence": oof_df[confidence_col].mean()
    })

metrics_df = pd.DataFrame(metric_rows)

macro_auc = metrics_df["ROC_AUC"].mean()
median_auc = metrics_df["ROC_AUC"].median()
minimum_auc = metrics_df["ROC_AUC"].min()

display(
    metrics_df.style.format({
        "ROC_AUC": "{:.4f}",
        "MeanScore_Gold0": "{:.4f}",
        "MeanScore_Gold1": "{:.4f}",
        "MeanConfidence": "{:.4f}"
    })
)

print(f"\nMacro ROC-AUC:  {macro_auc:.4f}")
print(f"Median ROC-AUC: {median_auc:.4f}")
print(f"Lowest ROC-AUC: {minimum_auc:.4f}")

print("\nTargets with AUC ≥ 0.800:")
print(
    metrics_df.loc[
        metrics_df["ROC_AUC"] >= 0.800,
        ["Target", "ROC_AUC"]
    ].to_string(index=False)
)

print("\nTargets with AUC < 0.800:")
print(
    metrics_df.loc[
        metrics_df["ROC_AUC"] < 0.800,
        ["Target", "ROC_AUC"]
    ].to_string(index=False)
)

METRICS_FILE = OUTPUT_DIR / "soft_label_v1_oof_metrics.csv"
metrics_df.to_csv(METRICS_FILE, index=False)

SUMMARY_FILE = OUTPUT_DIR / "soft_label_v1_oof_summary.json"

with open(SUMMARY_FILE, "w") as file:
    json.dump(
        {
            "model": MODEL_NAME,
            "n_gold_reports": len(oof_df),
            "n_folds": N_FOLDS,
            "n_demonstrations": N_DEMONSTRATIONS,
            "macro_roc_auc": float(macro_auc),
            "median_roc_auc": float(median_auc),
            "minimum_roc_auc": float(minimum_auc)
        },
        file,
        indent=2
    )

print("\nSaved:", METRICS_FILE)
print("Saved:", SUMMARY_FILE)
print("Soft Label V1 OOF evaluation complete ✓")

,Target,Positive,Negative,ROC_AUC,MeanScore_Gold0,MeanScore_Gold1,MeanConfidence
0,ACL,24,34,0.9638,0.1829,0.9800,0.9559
1,MCL,9,49,0.9717,0.1782,0.9800,0.9397
2,Medial Meniscus,26,32,0.9459,0.1691,0.8842,0.9564
3,Lateral Meniscus,23,35,0.8969,0.1563,0.7996,0.9200
4,Medial OA,15,43,0.9372,0.1560,0.9107,0.9033
5,Lateral OA,11,47,0.8627,0.1836,0.8036,0.8862
6,PF OA,21,37,0.9067,0.2257,0.7976,0.9129
7,Effusion,35,23,0.7913,0.6500,0.9071,0.9705
8,Synovitis,27,31,0.6983,0.3419,0.5904,0.7738
9,Baker's,12,46,0.9112,0.1728,0.9075,0.7757



Macro ROC-AUC:  0.8841
Median ROC-AUC: 0.9018
Lowest ROC-AUC: 0.6983

Targets with AUC ≥ 0.800:
          Target  ROC_AUC
             ACL 0.963848
             MCL 0.971655
 Medial Meniscus 0.945913
Lateral Meniscus 0.896894
       Medial OA 0.937209
      Lateral OA 0.862669
           PF OA 0.906692
         Baker's 0.911232
       Contusion 0.858974
        Fracture 0.864583

Targets with AUC < 0.800:
   Target  ROC_AUC
 Effusion 0.791304
Synovitis 0.698327

Saved: /kaggle/working/llm_pseudo_labels_v2/soft_label_v1_oof_metrics.csv
Saved: /kaggle/working/llm_pseudo_labels_v2/soft_label_v1_oof_summary.json
Soft Label V1 OOF evaluation complete ✓


In [11]:
# ============================================================
# Step 11: Audit weak-target OOF errors
# ============================================================

WEAK_TARGETS = ["Effusion", "Synovitis"]
N_AUDIT_CASES = 6

gold_report_lookup = (
    gold_df[
        ["StudyInstanceUID", "Report"]
    ]
    .assign(
        StudyInstanceUID=lambda x:
            x["StudyInstanceUID"].astype(str)
    )
)

audit_df = oof_df.copy()
audit_df["StudyInstanceUID"] = (
    audit_df["StudyInstanceUID"].astype(str)
)

audit_df = audit_df.merge(
    gold_report_lookup,
    on="StudyInstanceUID",
    how="left",
    validate="one_to_one"
)

assert audit_df["Report"].notna().all()


for target in WEAK_TARGETS:
    gold_col = f"{target}_gold"
    score_col = f"{target}_score"
    confidence_col = f"{target}_confidence"
    evidence_col = f"{target}_evidence"

    negatives = (
        audit_df[audit_df[gold_col] == 0]
        .sort_values(score_col, ascending=False)
        .head(N_AUDIT_CASES)
    )

    positives = (
        audit_df[audit_df[gold_col] == 1]
        .sort_values(score_col, ascending=True)
        .head(N_AUDIT_CASES)
    )

    print("\n" + "=" * 90)
    print(f"{target}: HIGHEST-SCORED GOLD NEGATIVES")
    print("=" * 90)

    for _, row in negatives.iterrows():
        print(
            f"\nStudy: {row['StudyInstanceUID']}\n"
            f"Fold: {int(row['fold'])}\n"
            f"Gold: 0 | Score: {row[score_col]:.3f} | "
            f"Confidence: {row[confidence_col]:.3f}\n"
            f"Evidence: {row[evidence_col]}\n"
            f"Report: {str(row['Report'])[:700]}"
        )
        print("-" * 90)

    print("\n" + "=" * 90)
    print(f"{target}: LOWEST-SCORED GOLD POSITIVES")
    print("=" * 90)

    for _, row in positives.iterrows():
        print(
            f"\nStudy: {row['StudyInstanceUID']}\n"
            f"Fold: {int(row['fold'])}\n"
            f"Gold: 1 | Score: {row[score_col]:.3f} | "
            f"Confidence: {row[confidence_col]:.3f}\n"
            f"Evidence: {row[evidence_col]}\n"
            f"Report: {str(row['Report'])[:700]}"
        )
        print("-" * 90)


Effusion: HIGHEST-SCORED GOLD NEGATIVES

Study: 1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269
Fold: 3
Gold: 0 | Score: 0.990 | Confidence: 0.990
Evidence: Moderate joint effusion with distention of the suprapatellar bursa is explicit.
Report:  The study reveals normal knee joint alignment.   No fracture is seen.  
   ACL is intact.   PCL is preserved.  
  The MCL is intact.    Medial meniscus is not torn.  
  The FCL, popliteus and biceps femoris tendons are preserved.       
   Horizontal tear at anterior horn of the lateral meniscus is noted.  
   Focal osteochondral defect at lateral patellar facet, about 8x12mm. with subchondral bone edema is detected.   Full thickness cartilage defect, 1.2x1.5cm. at lateral trochlea with subchondral bone edema is noted.   Small osteochondral body, about intact9xintact4cm. at anterior infrapatellar recess is noted.   
  The quadriceps and patellar tendons are preserved.   
   Moderate joint e
------------------------------------

In [12]:
# ============================================================
# Step 12: Validate GPT-5.6 Terra on the 58 OOF reports
# ============================================================

TERRA_MODEL = "gpt-5.6-terra"
TERRA_OOF_FILE = OUTPUT_DIR / "soft_label_v1_terra_oof_predictions.csv"


if TERRA_OOF_FILE.exists():
    terra_results = pd.read_csv(
        TERRA_OOF_FILE
    ).to_dict("records")

    terra_completed_uids = {
        str(row["StudyInstanceUID"])
        for row in terra_results
    }

    print(
        "Resuming Terra predictions:",
        len(terra_completed_uids)
    )

else:
    terra_results = []
    terra_completed_uids = set()

    print("Starting new Terra OOF validation")


for _, row in tqdm(
    gold_df.iterrows(),
    total=len(gold_df),
    desc="Terra OOF soft labels"
):
    uid = str(row["StudyInstanceUID"])

    if uid in terra_completed_uids:
        continue

    fold = int(row["fold"])
    last_error = None

    for attempt in range(3):
        try:
            response = client.responses.parse(
                model=TERRA_MODEL,
                reasoning={"effort": "low"},
                input=[
                    {
                        "role": "system",
                        "content": SOFT_LABEL_SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": build_oof_prompt(
                            report=row["Report"],
                            fold=fold
                        )
                    }
                ],
                text_format=SoftLabelPrediction
            )

            prediction = response.output_parsed

            assert prediction is not None
            assert len(prediction.predictions) == 12

            result_row = prediction_to_row(
                row,
                prediction
            )

            # Correct the model field set by the earlier helper
            result_row["model"] = TERRA_MODEL

            terra_results.append(result_row)
            terra_completed_uids.add(uid)

            # Save immediately after every paid request
            pd.DataFrame(terra_results).to_csv(
                TERRA_OOF_FILE,
                index=False
            )

            break

        except Exception as error:
            last_error = error
            wait_seconds = 2 ** attempt

            print(
                f"\nAttempt {attempt + 1}/3 failed "
                f"for {uid}: {error}"
            )

            time.sleep(wait_seconds)

    else:
        raise RuntimeError(
            f"Terra failed after 3 attempts for {uid}: "
            f"{last_error}\n"
            f"Completed results remain saved in "
            f"{TERRA_OOF_FILE}"
        )


terra_oof_df = pd.read_csv(TERRA_OOF_FILE)

assert len(terra_oof_df) == 58
assert (
    terra_oof_df["StudyInstanceUID"]
    .astype(str)
    .nunique()
    == 58
)

print("\nTerra OOF predictions:", len(terra_oof_df))
print(
    "Unique studies:",
    terra_oof_df["StudyInstanceUID"].nunique()
)
print("Saved:", TERRA_OOF_FILE)
print("GPT-5.6 Terra OOF validation completed ✓")

Starting new Terra OOF validation


Terra OOF soft labels:   0%|          | 0/58 [00:00<?, ?it/s]


Terra OOF predictions: 58
Unique studies: 58
Saved: /kaggle/working/llm_pseudo_labels_v2/soft_label_v1_terra_oof_predictions.csv
GPT-5.6 Terra OOF validation completed ✓


In [13]:
# ============================================================
# Step 13: Compare GPT-5.6 Terra with original GPT-5.6
# ============================================================

comparison_rows = []

for target in TARGET_COLS:
    gold_col = f"{target}_gold"
    score_col = f"{target}_score"

    original_auc = roc_auc_score(
        oof_df[gold_col].astype(int),
        oof_df[score_col].astype(float)
    )

    terra_auc = roc_auc_score(
        terra_oof_df[gold_col].astype(int),
        terra_oof_df[score_col].astype(float)
    )

    comparison_rows.append({
        "Target": target,
        "Original_AUC": original_auc,
        "Terra_AUC": terra_auc,
        "Terra_minus_Original": terra_auc - original_auc
    })


model_comparison_df = pd.DataFrame(comparison_rows)

original_macro_auc = (
    model_comparison_df["Original_AUC"].mean()
)

terra_macro_auc = (
    model_comparison_df["Terra_AUC"].mean()
)

macro_difference = (
    terra_macro_auc - original_macro_auc
)

display(
    model_comparison_df.style.format({
        "Original_AUC": "{:.4f}",
        "Terra_AUC": "{:.4f}",
        "Terra_minus_Original": "{:+.4f}"
    }).background_gradient(
        subset=["Terra_minus_Original"],
        cmap="RdYlGn",
        vmin=-0.10,
        vmax=0.10
    )
)

print(f"\nOriginal macro AUC: {original_macro_auc:.4f}")
print(f"Terra macro AUC:    {terra_macro_auc:.4f}")
print(f"Difference:         {macro_difference:+.4f}")

print("\nTerra targets with AUC ≥ 0.800:")
print(
    model_comparison_df.loc[
        model_comparison_df["Terra_AUC"] >= 0.800,
        ["Target", "Terra_AUC"]
    ].to_string(index=False)
)

print("\nTerra targets with AUC < 0.800:")
print(
    model_comparison_df.loc[
        model_comparison_df["Terra_AUC"] < 0.800,
        ["Target", "Terra_AUC"]
    ].to_string(index=False)
)

COMPARISON_FILE = (
    OUTPUT_DIR /
    "soft_label_v1_original_vs_terra.csv"
)

model_comparison_df.to_csv(
    COMPARISON_FILE,
    index=False
)

print("\nSaved:", COMPARISON_FILE)
print("Model comparison complete ✓")

,Target,Original_AUC,Terra_AUC,Terra_minus_Original
0,ACL,0.9638,0.9865,+0.0227
1,MCL,0.9717,0.9717,+0.0000
2,Medial Meniscus,0.9459,0.9315,-0.0144
3,Lateral Meniscus,0.8969,0.8888,-0.0081
4,Medial OA,0.9372,0.9457,+0.0085
5,Lateral OA,0.8627,0.8453,-0.0174
6,PF OA,0.9067,0.9003,-0.0064
7,Effusion,0.7913,0.8093,+0.0180
8,Synovitis,0.6983,0.7658,+0.0675
9,Baker's,0.9112,0.9058,-0.0054



Original macro AUC: 0.8841
Terra macro AUC:    0.8877
Difference:         +0.0036

Terra targets with AUC ≥ 0.800:
          Target  Terra_AUC
             ACL   0.986520
             MCL   0.971655
 Medial Meniscus   0.931490
Lateral Meniscus   0.888820
       Medial OA   0.945736
      Lateral OA   0.845261
           PF OA   0.900257
        Effusion   0.809317
         Baker's   0.905797
       Contusion   0.844804
        Fracture   0.856944

Terra targets with AUC < 0.800:
   Target  Terra_AUC
Synovitis    0.76583

Saved: /kaggle/working/llm_pseudo_labels_v2/soft_label_v1_original_vs_terra.csv
Model comparison complete ✓


In [14]:
# ============================================================
# Step 14: Build final Terra production context
# ============================================================

production_demonstrations, production_covered_states = (
    select_fold_demonstrations(
        gold_df,
        n_examples=N_DEMONSTRATIONS
    )
)

assert len(production_demonstrations) == N_DEMONSTRATIONS
assert len(production_covered_states) == 24

PRODUCTION_DEMO_UIDS = set(
    production_demonstrations[
        "StudyInstanceUID"
    ].astype(str)
)

PRODUCTION_DEMONSTRATION_TEXT = "\n\n".join(
    format_gold_demonstration(row)
    for _, row in production_demonstrations.iterrows()
)


def build_production_prompt(report):
    return f"""
The following are gold-labelled reference examples from the same
dataset. They demonstrate its dataset-specific annotation convention.

{PRODUCTION_DEMONSTRATION_TEXT}

------------------------------------------------------------

REPORT TO ANALYSE:

{str(report).strip()}

Analyse only the REPORT TO ANALYSE.

Return exactly one prediction for every target in TARGET_COLS.
Scores must remain continuous probabilities rather than hard labels.
Do not copy the overall label pattern from any reference example.
"""


# Save the selected production examples
PRODUCTION_DEMO_FILE = (
    OUTPUT_DIR /
    "soft_label_v1_production_demonstrations.csv"
)

production_demonstrations[
    ["StudyInstanceUID", "Report"] + TARGET_COLS
].to_csv(
    PRODUCTION_DEMO_FILE,
    index=False
)


# Verify one unlabelled production prompt
production_test_row = unlabelled_df.iloc[0]

production_test_prompt = build_production_prompt(
    production_test_row["Report"]
)

assert (
    str(production_test_row["StudyInstanceUID"])
    not in PRODUCTION_DEMO_UIDS
)
assert "REPORT TO ANALYSE:" in production_test_prompt

print(
    "Production demonstrations:",
    len(production_demonstrations)
)
print(
    "Target-label states covered:",
    len(production_covered_states),
    "/24"
)
print(
    "Test prompt characters:",
    len(production_test_prompt)
)
print("Saved:", PRODUCTION_DEMO_FILE)
print("Final Terra production context ready ✓")

Production demonstrations: 10
Target-label states covered: 24 /24
Test prompt characters: 6944
Saved: /kaggle/working/llm_pseudo_labels_v2/soft_label_v1_production_demonstrations.csv
Final Terra production context ready ✓


In [15]:
# ============================================================
# Step 15: Select a diverse unlabelled pilot sample
# ============================================================

import re

PILOT_SIZE = 12
PILOT_RANDOM_STATE = 42


def detect_script_group(text):
    text = str(text)

    if re.search(r"[\u0400-\u04FF]", text):
        return "Cyrillic"

    if re.search(r"[\u0370-\u03FF]", text):
        return "Greek"

    if re.search(r"[\u0600-\u06FF]", text):
        return "Arabic"

    if re.search(r"[\u4E00-\u9FFF]", text):
        return "CJK"

    return "Latin_or_other"


pilot_pool = unlabelled_df.copy()

pilot_pool["ReportCharacters"] = (
    pilot_pool["Report"].astype(str).str.len()
)

pilot_pool["ScriptGroup"] = (
    pilot_pool["Report"].apply(detect_script_group)
)

# Create report-length groups
pilot_pool["LengthGroup"] = pd.qcut(
    pilot_pool["ReportCharacters"],
    q=6,
    labels=[
        "Very short",
        "Short",
        "Medium-short",
        "Medium-long",
        "Long",
        "Very long"
    ],
    duplicates="drop"
)

selected_indices = []

# First select one report from each available script group
for script_group, group in pilot_pool.groupby("ScriptGroup"):
    chosen = group.sample(
        n=1,
        random_state=PILOT_RANDOM_STATE
    )

    selected_indices.extend(chosen.index.tolist())

# Fill the remaining positions across length groups
for length_group, group in pilot_pool.groupby(
    "LengthGroup",
    observed=True
):
    if len(selected_indices) >= PILOT_SIZE:
        break

    available = group.loc[
        ~group.index.isin(selected_indices)
    ]

    if len(available) > 0:
        chosen = available.sample(
            n=1,
            random_state=PILOT_RANDOM_STATE
        )

        selected_indices.extend(chosen.index.tolist())

# Fill any remaining positions randomly
remaining_needed = PILOT_SIZE - len(selected_indices)

if remaining_needed > 0:
    available = pilot_pool.loc[
        ~pilot_pool.index.isin(selected_indices)
    ]

    extra = available.sample(
        n=remaining_needed,
        random_state=PILOT_RANDOM_STATE
    )

    selected_indices.extend(extra.index.tolist())

pilot_df = (
    pilot_pool.loc[selected_indices]
    .head(PILOT_SIZE)
    .copy()
    .reset_index(drop=True)
)

assert len(pilot_df) == PILOT_SIZE
assert pilot_df["StudyInstanceUID"].is_unique

print("Pilot studies:", len(pilot_df))

print("\nScript coverage:")
print(pilot_df["ScriptGroup"].value_counts())

print("\nLength coverage:")
print(
    pilot_df[
        ["ScriptGroup", "LengthGroup", "ReportCharacters"]
    ].to_string(index=False)
)

print(
    "\nPilot report length range:",
    pilot_df["ReportCharacters"].min(),
    "to",
    pilot_df["ReportCharacters"].max()
)

print("\nDiverse production pilot selected ✓")

Pilot studies: 12

Script coverage:
ScriptGroup
Latin_or_other    8
Cyrillic          2
Greek             2
Name: count, dtype: int64

Length coverage:
   ScriptGroup  LengthGroup  ReportCharacters
      Cyrillic         Long              1566
         Greek  Medium-long              1039
Latin_or_other        Short               533
Latin_or_other   Very short               325
      Cyrillic        Short               695
Latin_or_other Medium-short               827
         Greek  Medium-long              1026
Latin_or_other         Long              1341
Latin_or_other    Very long              1916
Latin_or_other  Medium-long              1233
Latin_or_other Medium-short               824
Latin_or_other         Long              1486

Pilot report length range: 325 to 1916

Diverse production pilot selected ✓


In [16]:
# ============================================================
# Step 16: Generate 12 Terra production pilot predictions
# ============================================================

PILOT_FILE = (
    OUTPUT_DIR /
    "soft_label_v1_terra_production_pilot.csv"
)


def production_prediction_to_row(study_row, prediction):
    output_row = {
        "StudyInstanceUID": str(
            study_row["StudyInstanceUID"]
        ),
        "model": TERRA_MODEL,
        "ScriptGroup": study_row["ScriptGroup"],
        "LengthGroup": str(study_row["LengthGroup"]),
        "ReportCharacters": int(
            study_row["ReportCharacters"]
        )
    }

    prediction_map = {
        item.target: item
        for item in prediction.predictions
    }

    assert set(prediction_map) == set(TARGET_COLS)

    for target in TARGET_COLS:
        item = prediction_map[target]

        output_row[f"{target}_score"] = float(item.score)
        output_row[f"{target}_confidence"] = float(
            item.confidence
        )
        output_row[f"{target}_evidence"] = item.evidence

    return output_row


if PILOT_FILE.exists():
    pilot_results = pd.read_csv(
        PILOT_FILE
    ).to_dict("records")

    pilot_completed_uids = {
        str(row["StudyInstanceUID"])
        for row in pilot_results
    }

    print(
        "Resuming pilot predictions:",
        len(pilot_completed_uids)
    )

else:
    pilot_results = []
    pilot_completed_uids = set()


for _, row in tqdm(
    pilot_df.iterrows(),
    total=len(pilot_df),
    desc="Terra production pilot"
):
    uid = str(row["StudyInstanceUID"])

    if uid in pilot_completed_uids:
        continue

    last_error = None

    for attempt in range(3):
        try:
            response = client.responses.parse(
                model=TERRA_MODEL,
                reasoning={"effort": "low"},
                input=[
                    {
                        "role": "system",
                        "content": SOFT_LABEL_SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": build_production_prompt(
                            row["Report"]
                        )
                    }
                ],
                text_format=SoftLabelPrediction
            )

            prediction = response.output_parsed

            assert prediction is not None
            assert len(prediction.predictions) == 12

            result_row = production_prediction_to_row(
                row,
                prediction
            )

            pilot_results.append(result_row)
            pilot_completed_uids.add(uid)

            pd.DataFrame(pilot_results).to_csv(
                PILOT_FILE,
                index=False
            )

            break

        except Exception as error:
            last_error = error
            wait_seconds = 2 ** attempt

            print(
                f"\nAttempt {attempt + 1}/3 failed "
                f"for {uid}: {error}"
            )

            time.sleep(wait_seconds)

    else:
        raise RuntimeError(
            f"Pilot failed after 3 attempts for {uid}: "
            f"{last_error}"
        )


production_pilot_df = pd.read_csv(PILOT_FILE)

assert len(production_pilot_df) == PILOT_SIZE
assert (
    production_pilot_df["StudyInstanceUID"]
    .astype(str)
    .nunique()
    == PILOT_SIZE
)

score_cols = [
    f"{target}_score"
    for target in TARGET_COLS
]

confidence_cols = [
    f"{target}_confidence"
    for target in TARGET_COLS
]

assert (
    production_pilot_df[score_cols]
    .apply(lambda column: column.between(0, 1).all())
    .all()
)

assert (
    production_pilot_df[confidence_cols]
    .apply(lambda column: column.between(0, 1).all())
    .all()
)

print("\nPilot predictions:", len(production_pilot_df))
print(
    "Mean confidence:",
    production_pilot_df[confidence_cols]
    .to_numpy()
    .mean()
)
print(
    "Predictions with confidence < 0.50:",
    int(
        (
            production_pilot_df[confidence_cols]
            .to_numpy()
            < 0.50
        ).sum()
    )
)
print("Saved:", PILOT_FILE)
print("Terra production pilot completed ✓")

Terra production pilot:   0%|          | 0/12 [00:00<?, ?it/s]


Pilot predictions: 12
Mean confidence: 0.8809027777777777
Predictions with confidence < 0.50: 11
Saved: /kaggle/working/llm_pseudo_labels_v2/soft_label_v1_terra_production_pilot.csv
Terra production pilot completed ✓


In [17]:
# ============================================================
# Step 17: Audit low-confidence production pilot predictions
# ============================================================

pilot_report_lookup = (
    pilot_df[
        ["StudyInstanceUID", "Report"]
    ]
    .assign(
        StudyInstanceUID=lambda x:
            x["StudyInstanceUID"].astype(str)
    )
)

pilot_audit_df = production_pilot_df.copy()

pilot_audit_df["StudyInstanceUID"] = (
    pilot_audit_df["StudyInstanceUID"].astype(str)
)

pilot_audit_df = pilot_audit_df.merge(
    pilot_report_lookup,
    on="StudyInstanceUID",
    how="left",
    validate="one_to_one"
)

low_confidence_rows = []

for _, row in pilot_audit_df.iterrows():
    for target in TARGET_COLS:
        confidence = float(
            row[f"{target}_confidence"]
        )

        if confidence < 0.50:
            low_confidence_rows.append({
                "StudyInstanceUID": row["StudyInstanceUID"],
                "ScriptGroup": row["ScriptGroup"],
                "Target": target,
                "Score": float(row[f"{target}_score"]),
                "Confidence": confidence,
                "Evidence": row[f"{target}_evidence"],
                "Report": row["Report"]
            })

low_confidence_df = (
    pd.DataFrame(low_confidence_rows)
    .sort_values("Confidence")
    .reset_index(drop=True)
)

print(
    "Low-confidence target predictions:",
    len(low_confidence_df)
)

print("\nLow-confidence counts by target:")
print(
    low_confidence_df["Target"]
    .value_counts()
    .to_string()
)

print("\nLow-confidence counts by script:")
print(
    low_confidence_df["ScriptGroup"]
    .value_counts()
    .to_string()
)

for _, row in low_confidence_df.iterrows():
    print("\n" + "=" * 90)
    print("Study:", row["StudyInstanceUID"])
    print("Script:", row["ScriptGroup"])
    print("Target:", row["Target"])
    print(
        f"Score: {row['Score']:.3f} | "
        f"Confidence: {row['Confidence']:.3f}"
    )
    print("Evidence:", row["Evidence"])
    print("Report:", str(row["Report"])[:700])

print("\nLow-confidence pilot audit complete ✓")

Low-confidence target predictions: 11

Low-confidence counts by target:
Target
Baker's       7
Synovitis     2
PF OA         1
Lateral OA    1

Low-confidence counts by script:
ScriptGroup
Latin_or_other    7
Greek             3
Cyrillic          1

Study: 1.2.826.0.1.3680043.8.498.65607807464576681752987746458209389847
Script: Cyrillic
Target: Baker's
Score: 0.120 | Confidence: 0.300
Evidence: No Baker cyst is mentioned.
Report: МР находка: МР данни за стрес фрактура под главата на фибулата. Няма данни за дислокация. Изобразява се фрактурна линия през напречника на костта, около която се вижда изразен оток на костния мозък.  Тибия, патела – с нормална форма. Наличие на малък контузионен костно-мозъчен едем в медиалния кондил на фемура. Няма данни за ставен излив. Нормално изобразяване на менискусите. Кръстните връзки са със запазена цялост. Ставният хрущял е интактен. Колатерални лигаменти – без особености. Ретинакулуми – б.о. Лигамент на пателата, сухожилие на м. квадрицепс феморис –

In [ ]:
# ============================================================
# Step 18: Generate Terra soft labels for all 4,349 reports
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import random

FULL_JSONL_FILE = (
    OUTPUT_DIR /
    "soft_label_v1_terra_4349_checkpoint.jsonl"
)

FULL_CSV_FILE = (
    OUTPUT_DIR /
    "soft_label_v1_terra_4349.csv"
)

MAX_WORKERS = 5
MAX_RETRIES = 6

thread_local = threading.local()


def get_thread_client():
    if not hasattr(thread_local, "client"):
        thread_local.client = OpenAI(api_key=api_key)

    return thread_local.client


def predict_production_report(row_dict):
    uid = str(row_dict["StudyInstanceUID"])
    worker_client = get_thread_client()
    last_error = None

    for attempt in range(MAX_RETRIES):
        try:
            response = worker_client.responses.parse(
                model=TERRA_MODEL,
                reasoning={"effort": "low"},
                input=[
                    {
                        "role": "system",
                        "content": SOFT_LABEL_SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": build_production_prompt(
                            row_dict["Report"]
                        )
                    }
                ],
                text_format=SoftLabelPrediction
            )

            prediction = response.output_parsed

            assert prediction is not None
            assert len(prediction.predictions) == 12

            result = production_prediction_to_row(
                pd.Series(row_dict),
                prediction
            )

            return {
                "success": True,
                "uid": uid,
                "result": result
            }

        except Exception as error:
            last_error = str(error)

            wait_seconds = min(
                60,
                (2 ** attempt) + random.random()
            )

            time.sleep(wait_seconds)

    return {
        "success": False,
        "uid": uid,
        "error": last_error
    }


# ------------------------------------------------------------
# Load existing JSONL checkpoint
# ------------------------------------------------------------

results_by_uid = {}

if FULL_JSONL_FILE.exists():
    with open(
        FULL_JSONL_FILE,
        "r",
        encoding="utf-8"
    ) as file:
        for line in file:
            try:
                result = json.loads(line)
                uid = str(result["StudyInstanceUID"])
                results_by_uid[uid] = result

            except Exception:
                # Safely ignore an incomplete final line
                continue

    print(
        "Loaded checkpoint predictions:",
        len(results_by_uid)
    )

else:
    # Seed checkpoint with the 12 completed pilot results
    with open(
        FULL_JSONL_FILE,
        "w",
        encoding="utf-8"
    ) as file:
        for result in pilot_results:
            uid = str(result["StudyInstanceUID"])

            if uid not in results_by_uid:
                file.write(
                    json.dumps(
                        result,
                        ensure_ascii=False
                    ) + "\n"
                )

                results_by_uid[uid] = result

        file.flush()
        os.fsync(file.fileno())

    print(
        "Seeded checkpoint with pilot predictions:",
        len(results_by_uid)
    )


# ------------------------------------------------------------
# Prepare remaining unlabelled reports
# ------------------------------------------------------------

full_production_pool = pilot_pool.copy()

pending_rows = []

for _, row in full_production_pool.iterrows():
    uid = str(row["StudyInstanceUID"])

    if uid not in results_by_uid:
        pending_rows.append(row.to_dict())

print("Total unlabelled reports:", len(full_production_pool))
print("Already completed:", len(results_by_uid))
print("Remaining API calls:", len(pending_rows))
print("Concurrent workers:", MAX_WORKERS)


# ------------------------------------------------------------
# Run concurrent synchronous API requests
# ------------------------------------------------------------

failed_uids = []

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    future_to_uid = {
        executor.submit(
            predict_production_report,
            row_dict
        ): str(row_dict["StudyInstanceUID"])
        for row_dict in pending_rows
    }

    with open(
        FULL_JSONL_FILE,
        "a",
        encoding="utf-8"
    ) as checkpoint_file:

        for future in tqdm(
            as_completed(future_to_uid),
            total=len(future_to_uid),
            desc="Terra full production"
        ):
            uid = future_to_uid[future]

            try:
                outcome = future.result()

            except Exception as error:
                failed_uids.append(uid)
                print(f"\nUnexpected failure for {uid}: {error}")
                continue

            if not outcome["success"]:
                failed_uids.append(uid)
                print(
                    f"\nFailed after retries for {uid}: "
                    f"{outcome['error']}"
                )
                continue

            result = outcome["result"]
            result_uid = str(result["StudyInstanceUID"])

            if result_uid not in results_by_uid:
                checkpoint_file.write(
                    json.dumps(
                        result,
                        ensure_ascii=False
                    ) + "\n"
                )

                checkpoint_file.flush()
                os.fsync(checkpoint_file.fileno())

                results_by_uid[result_uid] = result


# ------------------------------------------------------------
# Build final CSV when complete
# ------------------------------------------------------------

print("\nCompleted predictions:", len(results_by_uid))
print("Failed reports:", len(failed_uids))
print("Checkpoint:", FULL_JSONL_FILE)

if failed_uids:
    print(
        "\nSome reports failed. Rerun this same cell; "
        "only missing reports will be requested again."
    )

else:
    final_soft_df = pd.DataFrame(
        results_by_uid.values()
    )

    # Restore original unlabelled-study ordering
    original_order = {
        str(uid): position
        for position, uid in enumerate(
            unlabelled_df["StudyInstanceUID"]
        )
    }

    final_soft_df["_order"] = (
        final_soft_df["StudyInstanceUID"]
        .astype(str)
        .map(original_order)
    )

    final_soft_df = (
        final_soft_df
        .sort_values("_order")
        .drop(columns="_order")
        .reset_index(drop=True)
    )

    assert len(final_soft_df) == 4349
    assert (
        final_soft_df["StudyInstanceUID"]
        .astype(str)
        .nunique()
        == 4349
    )

    final_soft_df.to_csv(
        FULL_CSV_FILE,
        index=False
    )

    print("\nFinal rows:", len(final_soft_df))
    print(
        "Unique studies:",
        final_soft_df["StudyInstanceUID"].nunique()
    )
    print("Saved:", FULL_CSV_FILE)
    print("All 4,349 Terra soft labels complete ✓")